## Import related functions

In [1]:
import cobra
import re 
import pandas as pd
#from script.ECMpy_function import *
import sys
sys.path.append(r'./script/')
from ECMpy_function import *

D:\pycharm\ecCG827\.venv\Lib\site-packages\bioservices\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


# Input and output files

In [2]:
#Initial parameters
f = 0.46 
ptot = 0.56 
sigma = 0.5
lowerbound = 0
upperbound = round(ptot * f * sigma, 3) 
EC_max_file='./data/EC_kcat_max.json'# https://www.brenda-enzymes.org/brenda_download/file_download.php
method='DLKcat'#DLKcat
use_substrate='EX_cpd00027_e'
concentration=5.1
obj='Bio_cgATCC14067_c'# CG_biomass_cgl_ATCC13032 EX_lys_L_e
#Originl ecmodel and result file
ecModel_file="./model/ecCZ870_%s.json"%method
fluxes_infile_ori = './get_kcat_mw_by_%s/ECMpy_solution_Bio_cgATCC14067_c_pfba.csv'%method
reaction_kcat_MW_file = "./get_kcat_mw_by_%s/reaction_kcat_MW.csv"%method
need_change_reaction_list=[]
changed_reaction_list=[]
round_num=1
reaction_kcat_mw = pd.read_csv(reaction_kcat_MW_file, index_col=0)
folder = "./model_Calibration/%s_Calibration"%method
create_file(folder)
#ecmodel and result file
json_output_file = './model_Calibration/%s_Calibration/ecCZ870_%s_Calibration.json'% (method , method)
reaction_kcat_MW_outfile = "./model_Calibration/%s_Calibration/reaction_kcat_MW.csv"%method

Path exists


# Calibration kcat according Enzyme usage 

In [ ]:
#Calibration
enz_model_obj=0
while (enz_model_obj<0.46 and round_num<51):#maxium growth rate
    [enz_model,reaction_kcat_mw,need_change_reaction_list, changed_reaction_list]=change_enz_model_by_enz_usage(ecModel_file,fluxes_infile_ori,EC_max_file,\
    reaction_kcat_mw,need_change_reaction_list,changed_reaction_list,f, \
    ptot, sigma, lowerbound, upperbound, json_output_file)
    enz_model.objective=obj
    #change original substrate in model
    [ori_obj_id,ori_substrate_id_list,ori_sub_concentration,ori_ATPM]=get_model_substrate_obj(enz_model)
    for eachsubid in ori_substrate_id_list:
        if re.search('_reverse',eachsubid):
            r_id_new=eachsubid.split('_reverse')[0]
            enz_model.reactions.get_by_id(eachsubid).bounds = (0, 0) 
            enz_model.reactions.get_by_id(r_id_new).bounds = (0, 0)  
        else:
            r_id_new=eachsubid+'_reverse'
            enz_model.reactions.get_by_id(eachsubid).bounds = (0, 0) 
            enz_model.reactions.get_by_id(r_id_new).bounds = (0, 0) 
            
    enz_model.reactions.get_by_id(use_substrate).bounds = (-concentration, 0)
    enz_model.reactions.get_by_id(use_substrate+'_reverse').bounds = (0, 0)

    enz_model_obj=enz_model.slim_optimize()
    print('Calibration round %s : '%round_num+str(enz_model_obj))
    round_num=round_num+1

reaction_kcat_mw.to_csv(reaction_kcat_MW_outfile)

Need changing reaction: 
rxn01000_c
Set parameter Username
Set parameter LicenseID to value 2723056
Academic license - for non-commercial use only - expires 2026-10-16
Original bounds for EX_cpd00027_e_reverse: (0.0, 5.1)
Set objective to: EX_biomass_c
  Rate 7: Growth = 0.12387398109782129
  Rate 8: Growth = 0.12387398109782086
  Rate 9: Growth = 0.12387398109782086
Validation - Rates: [7, 8, 9], Growth: [0.12387398109782129, 0.12387398109782086, 0.12387398109782086]
Changed reaction: 
['rxn01000_c']
Calibration round 1 : 0.12089724413510122
Need changing reaction: 
rxn08173_c


In [1]:
import cobra
import re
import pandas as pd
#from script.ECMpy_function import *
import sys
sys.path.append(r'./script/')
from ECMpy_function import *
# 定义要运行的方法列表
methods = ['AutoPACMEN_mean','AutoPACMEN_median','catapro','catpred','DLKcat','UniKP']
for method in methods:
    print(f"\n{'='*50}")
    print(f"开始运行方法: {method}")
    print(f"{'='*50}\n")

    # Initial parameters
    f = 0.46
    ptot = 0.56
    sigma = 0.5
    lowerbound = 0
    upperbound = round(ptot * f * sigma, 3)
    EC_max_file='./data/EC_kcat_max.json'
    use_substrate='EX_cpd00027_e'
    concentration=5.1
    obj='Bio_cgATCC14067_c'

    # 根据当前method设置文件路径
    ecModel_file = f"./model/ecCZ870_{method}.json"
    fluxes_infile_ori = f'./get_kcat_mw_by_{method}/ECMpy_solution_Bio_cgATCC14067_c_pfba.csv'
    reaction_kcat_MW_file = f"./get_kcat_mw_by_{method}/reaction_kcat_MW.csv"

    need_change_reaction_list = []
    changed_reaction_list = []
    round_num = 1
    reaction_kcat_mw = pd.read_csv(reaction_kcat_MW_file, index_col=0)

    folder = f"./model_Calibration/{method}_Calibration"
    create_file(folder)

    json_output_file = f'./model_Calibration/{method}_Calibration/ecCZ870_{method}_Calibration.json'
    reaction_kcat_MW_outfile = f"./model_Calibration/{method}_Calibration/reaction_kcat_MW.csv"

    # Calibration
    enz_model_obj = 0
    while (enz_model_obj < 0.46 and round_num < 51):
        [enz_model, reaction_kcat_mw, need_change_reaction_list, changed_reaction_list] = \
            change_enz_model_by_enz_usage(ecModel_file, fluxes_infile_ori, EC_max_file,
                                         reaction_kcat_mw, need_change_reaction_list,
                                         changed_reaction_list, f, ptot, sigma,
                                         lowerbound, upperbound, json_output_file)
        enz_model.objective = obj

        # change original substrate in model
        [ori_obj_id, ori_substrate_id_list, ori_sub_concentration, ori_ATPM] = \
            get_model_substrate_obj(enz_model)

        for eachsubid in ori_substrate_id_list:
            if re.search('_reverse', eachsubid):
                r_id_new = eachsubid.split('_reverse')[0]
                enz_model.reactions.get_by_id(eachsubid).bounds = (0, 0)
                enz_model.reactions.get_by_id(r_id_new).bounds = (0, 0)
            else:
                r_id_new = eachsubid + '_reverse'
                enz_model.reactions.get_by_id(eachsubid).bounds = (0, 0)
                enz_model.reactions.get_by_id(r_id_new).bounds = (0, 0)

        enz_model.reactions.get_by_id(use_substrate).bounds = (-concentration, 0)
        enz_model.reactions.get_by_id(use_substrate + '_reverse').bounds = (0, 0)

        enz_model_obj = enz_model.slim_optimize()
        print(f'Method {method} - Calibration round {round_num}: {enz_model_obj}')
        round_num = round_num + 1

    reaction_kcat_mw.to_csv(reaction_kcat_MW_outfile)
    print(f"\n方法 {method} 完成！\n")

D:\pycharm\ecCG827\.venv\Lib\site-packages\bioservices\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources



开始运行方法: AutoPACMEN_mean

Path exists
Need changing reaction: 
rxn01643_c
Set parameter Username
Set parameter LicenseID to value 2723056
Academic license - for non-commercial use only - expires 2026-10-16
Original bounds for EX_cpd00027_e_reverse: (0.0, 5.1)
Set objective to: EX_biomass_c
  Rate 7: Growth = 0.041391997098612396
  Rate 8: Growth = 0.04139199709861267
  Rate 9: Growth = 0.04139199709861267
Validation - Rates: [7, 8, 9], Growth: [0.041391997098612396, 0.04139199709861267, 0.04139199709861267]
Changed reaction: 
['rxn01643_c']
Method AutoPACMEN_mean - Calibration round 1: 0.04139198836723579
Need changing reaction: 
rxn01000_c
Original bounds for EX_cpd00027_e_reverse: (0.0, 5.1)
Set objective to: EX_biomass_c
  Rate 7: Growth = 0.07128687920926921
  Rate 8: Growth = 0.07128687920926852
  Rate 9: Growth = 0.07128687920926852
Validation - Rates: [7, 8, 9], Growth: [0.07128687920926921, 0.07128687920926852, 0.07128687920926852]
Changed reaction: 
['rxn01643_c', 'rxn01000_c'